# Exercise 1

Importing the libraries and loading the dataset

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv('/kaggle/input/bigdata2024classification/train.csv')
data.head()

,Id,Title,Content,Label
0,227464,"Netflix is coming to cable boxes, and Amazon i...",if you subscribe to one of three rinky-dink (...,Entertainment
1,244074,"Pharrell, Iranian President React to Tehran 'H...","pharrell, iranian president react to tehran '...",Entertainment
2,60707,Wildlife service seeks comments,the u.s. fish and wildlife service has reopen...,Technology
3,27883,Facebook teams up with Storyful to launch 'FB ...,the very nature of social media means it is o...,Technology
4,169596,Caesars plans US$880 mln New York casino,caesars plans us$880 mln new york casino jul ...,Business


Get some insights of the data

In [ ]:
print("\nColumn Data Types:")
display(data.dtypes)

print("\nMissing Values:")
display(data.isnull().sum())

print(f"Length of df {len(data)}")
value_counts = data['Label'].value_counts()
print(value_counts)



Column Data Types:


Id          int64
Title      object
Content    object
Label      object
dtype: object


Missing Values:


Id         0
Title      0
Content    0
Label      0
dtype: int64

Length of df 111795
Label
Entertainment    44834
Technology       30107
Business         24834
Health           12020
Name: count, dtype: int64


Since no missing vlaues, we will not have to make any pre-processing steps relating this.
The dataset is not perfectly balanced. In case this affects the final models, we will have to consider different techniques, such are undersampling or even generate synthetic data. We will see how it goes...

As a first step we will try how different pre-processing techniques work on our dataset. To do this, we will subsample the dataset and start trying different techniques incrementaly. For each step we will perform cross validation on the subsampled dataset. If there is an improvement in accuracy, we will consider this step in our pre-processing pipeline. First, we combine the title and the content of each row. Then we lowercase. Then we test the following :
- Remove punctuation
- Remove stopwords
- Apply either stemming or lemmatizaion
- Remove numbers
- Remove extra space

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import spacy
# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

# Step 1: Lowercase the text
def to_lowercase(text):
    return text.lower()

# Step 2: Remove punctuation
def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)

# Step 3: Remove stopwords
def remove_stopwords(text):
    stop_words = spacy.lang.en.stop_words.STOP_WORDS
    tokens = text.split()
    filtered = [word for word in tokens if word not in stop_words]
    return ' '.join(filtered)

# Step 4: Tokenize text
def tokenize(text):
    doc = nlp(text)
    return [token.text for token in doc]

# Step 5: Apply stemming (use spaCy's lemmatizer instead)
def apply_stemming(text):
    doc = nlp(text)
    stemmed = [token.lemma_ for token in doc]
    return ' '.join(stemmed)

# Step 6: Apply lemmatization
def apply_lemmatization(text):
    doc = nlp(text)
    lemmatized = [token.lemma_ for token in doc]
    return ' '.join(lemmatized)

# Step 7: Remove numbers
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

# Step 8: Combine Title and Content
def combine_features(row):
    return row['Title'] + ' ' + row['Content']

# Step 9: Remove extra whitespaces
def remove_extra_whitespace(text):
    return ' '.join(text.split())


We define a text preprocessing pipeline for natural language processing tasks.IncludING several functions to clean and transform the text data:

- **Lowercasing**: Converts text to lowercase for uniformity.
- **Removing Punctuation**: Strips punctuation marks to retain only words and spaces.
- **Removing Stopwords**:  Eliminates common words using spaCy’s stopword list.
- **Tokenization**:  Splits text into individual words using spaCy.
- **Stemming**:  Reduces words to their base form using spaCy’s lemmatizer.
- **Lemmatization**: Converts words to their dictionary root form.
- **Removing Numbers** Deletes numerical values from text.
- **Feature Combination**: Merges the "Title" and "Content" columns of a dataset.
- **Whitespace Removal**: Removes extra spaces to standardize text formatting.
We import essential libraries, including spaCy, NLTK, and scikit-learn, for text processing and machine learning. These functions are used to clean textual data before applying machine learning models.

In [ ]:
data_sample = data.sample(n=500, random_state=42)
data_sample['Text'] = data_sample.apply(combine_features, axis=1)
data_sample['Processed_Text'] = data_sample['Text'].apply(to_lowercase)

steps = [
    remove_punctuation,
    remove_stopwords,
    apply_stemming,
    remove_numbers,
    remove_extra_whitespace
]

results = []
for step in steps:
    print(f"Applying {step.__name__}")
    data_sample['Processed_Text'] = data_sample['Processed_Text'].apply(step)
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(data_sample['Processed_Text'])
    y = data_sample['Label']
    svm = SVC()
    svm_scores = cross_val_score(svm, X, y, cv=5, scoring='accuracy')
    rf = RandomForestClassifier()
    rf_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
    results.append({
        'Step': step.__name__,
        'SVM_Accuracy': svm_scores.mean(),
        'RF_Accuracy': rf_scores.mean()
    })

results_df = pd.DataFrame(results)


Applying remove_punctuation
Applying remove_stopwords
Applying apply_stemming
Applying remove_numbers
Applying remove_extra_whitespace


In [ ]:
results_df

,Step,SVM_Accuracy,RF_Accuracy
0,remove_punctuation,0.618,0.740
1,remove_stopwords,0.684,0.770
2,apply_stemming,0.716,0.768
3,remove_numbers,0.718,0.776
4,remove_extra_whitespace,0.718,0.768


Same for lemmatization

In [ ]:
data_sample = data.sample(n=500, random_state=42)
data_sample['Text'] = data_sample.apply(combine_features, axis=1)
data_sample['Processed_Text'] = data_sample['Text'].apply(to_lowercase)

steps = [
    remove_punctuation,
    remove_stopwords,
    apply_lemmatization,
    remove_numbers,
    remove_extra_whitespace
]

results = []
for step in steps:
    print(f"Applying {step.__name__}")
    data_sample['Processed_Text'] = data_sample['Processed_Text'].apply(step)
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(data_sample['Processed_Text'])
    y = data_sample['Label']
    svm = SVC()
    svm_scores = cross_val_score(svm, X, y, cv=5, scoring='accuracy')
    rf = RandomForestClassifier()
    rf_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
    results.append({
        'Step': step.__name__,
        'SVM_Accuracy': svm_scores.mean(),
        'RF_Accuracy': rf_scores.mean()
    })

results_df = pd.DataFrame(results)

Applying remove_punctuation
Applying remove_stopwords
Applying apply_lemmatization
Applying remove_numbers
Applying remove_extra_whitespace


In [ ]:
results_df

,Step,SVM_Accuracy,RF_Accuracy
0,remove_punctuation,0.618,0.736
1,remove_stopwords,0.684,0.768
2,apply_lemmatization,0.716,0.766
3,remove_numbers,0.718,0.778
4,remove_extra_whitespace,0.718,0.770


Everything helped. Lemmatization works slightly better for RF, we keep lemmatization.

Now we need to apply the pre-processing steps in the whole dataset. Since our dataset is large, we will consider spacy to use the available gpu and make the pre-processing faster. The functions change a bit, but ultimately we follow the same steps.

###  Preprocessing with GPU**
- **Utilizing GPU Acceleration**: The dataset is large, so `spaCy` is configured to use GPU for faster processing.
- **Feature Combination**: Merges the `Title` and `Content` columns into a single text field.
- **Dataset Conversion**: Converts the Pandas DataFrame into a Hugging Face `Dataset` for efficient processing.
- **Batch Processing**:
  - Uses `spaCy`'s `pipe()` function to process text in batches.
  - Applies **lemmatization** while removing stopwords, punctuation, and numbers.
- **Final Conversion**: The processed dataset is converted back into a Pandas DataFrame for further use.

---

In [ ]:
import spacy
from datasets import Dataset
import pandas as pd

# Enable GPU usage for SpaCy
spacy.require_gpu()

# Load the SpaCy model
nlp = spacy.load("en_core_web_sm")

# Combine features for the entire dataset
data['Text'] = data.apply(combine_features, axis=1)

# Convert the data to a Hugging Face Dataset
dataset = Dataset.from_pandas(data)

# Define the pre-processing function
def preprocess(batch):
    docs = list(nlp.pipe(batch["Text"], batch_size=32))
    processed_text = []
    for doc in docs:
        tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and not token.like_num]
        processed_text.append(" ".join(tokens))
    return {"Processed_Text": processed_text}

# Apply preprocessing in batches
processed_dataset = dataset.map(preprocess, batched=True, batch_size=32)

# Convert back to pandas DataFrame
data = processed_dataset.to_pandas()


/home/user/miniconda3/envs/high_scale/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 111795/111795 [25:36<00:00, 72.74 examples/s]


Same process for the test set

In [ ]:
import spacy
from datasets import Dataset
import pandas as pd

test_data = pd.read_csv('/kaggle/input/bigdata2024classification/test_without_labels.csv')
spacy.prefer_gpu()
nlp = spacy.load("en_core_web_sm")
test_data['Text'] = test_data.apply(combine_features, axis=1)

# Convert the data to a Hugging Face Dataset
dataset = Dataset.from_pandas(test_data)

# Define the pre-processing function
def preprocess(batch):
    docs = list(nlp.pipe(batch["Text"], batch_size=32))
    processed_text = []
    for doc in docs:
        tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and not token.like_num]
        processed_text.append(" ".join(tokens))
    return {"Processed_Text": processed_text}

# Apply preprocessing in batches
processed_dataset = dataset.map(preprocess, batched=True, batch_size=32)

# Convert back to pandas DataFrame
test_data = processed_dataset.to_pandas()


Map: 100%|██████████| 47912/47912 [11:22<00:00, 70.23 examples/s]


In [ ]:
data.head()

,Id,Title,Content,Label,Text,Processed_Text
0,227464,"Netflix is coming to cable boxes, and Amazon i...",if you subscribe to one of three rinky-dink (...,Entertainment,"Netflix is coming to cable boxes, and Amazon i...",Netflix come cable box Amazon grocery overlord...
1,244074,"Pharrell, Iranian President React to Tehran 'H...","pharrell, iranian president react to tehran '...",Entertainment,"Pharrell, Iranian President React to Tehran 'H...",Pharrell iranian President React Tehran Happy ...
2,60707,Wildlife service seeks comments,the u.s. fish and wildlife service has reopen...,Technology,Wildlife service seeks comments the u.s. fish...,wildlife service seek comment u.s fish wildl...
3,27883,Facebook teams up with Storyful to launch 'FB ...,the very nature of social media means it is o...,Technology,Facebook teams up with Storyful to launch 'FB ...,facebook team Storyful launch FB Newswire na...
4,169596,Caesars plans US$880 mln New York casino,caesars plans us$880 mln new york casino jul ...,Business,Caesars plans US$880 mln New York casino caes...,Caesars plan us$ mln New York casino caesar ...


In [ ]:
test_data.head()

,Id,Title,Content,Text,Processed_Text
0,262120,Tracy Morgan upgraded to fair condition after ...,actor and comedian tracy morgan has been upgr...,Tracy Morgan upgraded to fair condition after ...,Tracy Morgan upgrade fair condition crash ac...
1,175132,Smartphones Weigh on Samsung Electronics as Gu...,samsung electronics co ltd on tuesday issued u...,Smartphones Weigh on Samsung Electronics as Gu...,smartphone Weigh Samsung Electronics Guidance ...
2,218739,FBI denies fumbling testimony on 'X-Men' direc...,michael f. egan iii said in a press conferenc...,FBI denies fumbling testimony on 'X-Men' direc...,FBI deny fumble testimony X man director Bryan...
3,253483,Bachelorette 2014 Spoilers: Week 3 Recap ??? E...,i am having mixed emotions for what is about ...,Bachelorette 2014 Spoilers: Week 3 Recap ??? E...,Bachelorette spoiler week recap Eric Hill Dram...
4,224109,Barack Obama honours Frankie Knuckles in lette...,u.s. president barack obama has paid a specia...,Barack Obama honours Frankie Knuckles in lette...,barack Obama honour Frankie Knuckles letter lo...


Store the pre-processed datast as pickle files so we wont have to perform the pre-processing steps each time.

In [ ]:
data.to_pickle("/kaggle/input/pickle-files/processed_data.pkl")
test_data.to_pickle("/kaggle/input/pickle-files/processed_test_data.pkl")

In [ ]:
import pandas as pd

# Load the pickled DataFrames
data = pd.read_pickle("/kaggle/input/pickle-files/processed_data.pkl")
test_data = pd.read_pickle("/kaggle/input/pickle-files/processed_test_data.pkl")

Lets begin with training our models. This is a classification task. We will use the label encoder from sklearn to encode each class to numbers

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

data['Encoded_Label'] = label_encoder.fit_transform(data['Label'])

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label Mapping:", label_mapping)


Label Mapping: {'Business': 0, 'Entertainment': 1, 'Health': 2, 'Technology': 3}


Seperate the training and test sets.

In [ ]:
X_test = test_data['Processed_Text']
X = data['Processed_Text']
y = data['Encoded_Label']
print(X.shape)
print(y.shape)
print(X_test.shape)

(111795,)
(111795,)
(47912,)


In [ ]:
X.head()

0    Netflix come cable box Amazon grocery overlord...
1    Pharrell iranian President React Tehran Happy ...
2    wildlife service seek comment   u.s fish wildl...
3    facebook team Storyful launch FB Newswire   na...
4    Caesars plan us$ mln New York casino   caesar ...
Name: Processed_Text, dtype: object

In [ ]:
X_test.head()

0    Netflix come cable box Amazon grocery overlord...
1    Pharrell iranian President React Tehran Happy ...
2    wildlife service seek comment   u.s fish wildl...
3    facebook team Storyful launch FB Newswire   na...
4    Caesars plan us$ mln New York casino   caesar ...
Name: Processed_Text, dtype: object

There are hyperparameters to tune for bag of words, namely ngram range, max features and binary :
- ngram_range: Defines the range of n-grams (e.g., single words, pairs, or sequences of words) to extract from the text.
- max_features: Limits the number of most frequent tokens (words or n-grams) to consider, reducing dimensionality.
- binary: Determines whether to use binary presence (1 or 0) or term frequency (count) for token representation.

Also some hyperparameters for C :
- C: Regularization parameter controlling the trade-off between maximizing the margin and minimizing classification error.
- kernel: Specifies the type of kernel function used for transforming data (e.g., linear).

We optimize hyperparameters for both text preprocessing and model training using Optuna. For preprocessing with CountVectorizer, we explore n-grams, document frequency thresholds, and text cleaning options. The SVM optimization includes kernel selection (linear, rbf, poly), regularization strength, gamma, degree for polynomial kernel, and class weights. Random Forest parameters cover number of trees (100-1000), tree depth (5-50), min samples for splits/leaves, feature selection strategy, and bootstrap options. Using 500 samples and 5-fold cross-validation, we run 50 trials per model to maximize classification accuracy. Optuna's TPE sampler efficiently handles conditional parameters, like kernel-specific settings for SVM, and automatically prunes unpromising trials.

In [ ]:
import pandas as pd
import optuna
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Sample data (500 points)
data_sample = data.sample(n=500, random_state=42)

X_text = data_sample['Processed_Text']
y = data_sample['Encoded_Label']

# Enhanced vectorizer options
vectorizer_options = {
    "ngram_range": ["(1, 1)", "(1, 2)", "(1, 3)", "(1, 4)"],
    "max_features": [None, 10000],
    "binary": [False, True],
    "min_df": [1, 2, 5],
    "max_df": [0.9, 0.95, 1.0],
    "strip_accents": [None, 'unicode'],
    "lowercase": [True, False]
}

def svm_objective(trial):
    # Vectorizer parameters
    ngram_range_str = trial.suggest_categorical("ngram_range", vectorizer_options["ngram_range"])
    ngram_range = eval(ngram_range_str)  # Convert string back to tuple
    max_features = trial.suggest_categorical("max_features", vectorizer_options["max_features"])
    binary = trial.suggest_categorical("binary", vectorizer_options["binary"])
    min_df = trial.suggest_categorical("min_df", vectorizer_options["min_df"])
    max_df = trial.suggest_categorical("max_df", vectorizer_options["max_df"])
    strip_accents = trial.suggest_categorical("strip_accents", vectorizer_options["strip_accents"])
    lowercase = trial.suggest_categorical("lowercase", vectorizer_options["lowercase"])

    # Enhanced SVM parameters
    C = trial.suggest_float("C", 0.01, 10.0, log=True)
    kernel = trial.suggest_categorical("kernel", ["linear", "rbf", "poly"])

    # Additional SVM parameters based on kernel
    if kernel in ["rbf", "poly"]:
        gamma = trial.suggest_float("gamma", 1e-4, 1e1, log=True)
    else:
        gamma = "auto"

    if kernel == "poly":
        degree = trial.suggest_int("degree", 2, 5)
        coef0 = trial.suggest_float("coef0", 0.0, 10.0)
    else:
        degree = 3
        coef0 = 0.0

    # Class weights
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

    # Vectorize features
    vectorizer = CountVectorizer(
        ngram_range=ngram_range,
        max_features=max_features,
        binary=binary,
        min_df=min_df,
        max_df=max_df,
        strip_accents=strip_accents,
        lowercase=lowercase
    )
    X = vectorizer.fit_transform(X_text)

    # Train SVM with enhanced parameters
    svm = SVC(
        C=C,
        kernel=kernel,
        gamma=gamma,
        degree=degree,
        coef0=coef0,
        class_weight=class_weight,
        random_state=42
    )
    scores = cross_val_score(svm, X, y, cv=5, scoring="accuracy")
    return scores.mean()

def rf_objective(trial):
    # Vectorizer parameters
    ngram_range_str = trial.suggest_categorical("ngram_range", vectorizer_options["ngram_range"])
    ngram_range = eval(ngram_range_str)  # Convert string back to tuple
    max_features = trial.suggest_categorical("max_features", vectorizer_options["max_features"])
    binary = trial.suggest_categorical("binary", vectorizer_options["binary"])
    min_df = trial.suggest_categorical("min_df", vectorizer_options["min_df"])
    max_df = trial.suggest_categorical("max_df", vectorizer_options["max_df"])
    strip_accents = trial.suggest_categorical("strip_accents", vectorizer_options["strip_accents"])
    lowercase = trial.suggest_categorical("lowercase", vectorizer_options["lowercase"])

    # Enhanced RF parameters
    n_estimators = trial.suggest_int("n_estimators", 100, 1000)
    max_depth = trial.suggest_int("max_depth", 5, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features_rf = trial.suggest_categorical("max_features_rf", ["sqrt", "log2", None])
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"])

    # Vectorize features
    vectorizer = CountVectorizer(
        ngram_range=ngram_range,
        max_features=max_features,
        binary=binary,
        min_df=min_df,
        max_df=max_df,
        strip_accents=strip_accents,
        lowercase=lowercase
    )
    X = vectorizer.fit_transform(X_text)

    # Train RF with enhanced parameters
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features_rf,
        bootstrap=bootstrap,
        class_weight=class_weight,
        random_state=42
    )
    scores = cross_val_score(rf, X, y, cv=5, scoring="accuracy")
    return scores.mean()

# Run Optuna studies
svm_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
svm_study.optimize(svm_objective, n_trials=50)

rf_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
rf_study.optimize(rf_objective, n_trials=50)

# Display results
print("Best SVM Parameters:", svm_study.best_trial.params)
print("Best SVM Accuracy:", svm_study.best_trial.value)

print("Best RF Parameters:", rf_study.best_trial.params)
print("Best RF Accuracy:", rf_study.best_trial.value)


[I 2025-01-14 00:47:22,437] A new study created in memory with name: no-name-99d4401e-1418-4cea-a0a5-aefc883ba43c


[I 2025-01-14 00:47:24,108] Trial 0 finished with value: 0.21799999999999997 and parameters: {'ngram_range': '(1, 2)', 'max_features': 10000, 'binary': True, 'min_df': 1, 'max_df': 1.0, 'strip_accents': 'unicode', 'lowercase': False, 'C': 0.018986936485859817, 'kernel': 'poly', 'gamma': 2.0729536046024526, 'degree': 5, 'coef0': 8.741814080248737, 'class_weight': None}. Best is trial 0 with value: 0.21799999999999997.
[I 2025-01-14 00:47:26,365] Trial 1 finished with value: 0.42400000000000004 and parameters: {'ngram_range': '(1, 4)', 'max_features': 10000, 'binary': False, 'min_df': 2, 'max_df': 0.95, 'strip_accents': None, 'lowercase': True, 'C': 7.399406222118706, 'kernel': 'rbf', 'gamma': 1.008380445657876, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.42400000000000004.
[I 2025-01-14 00:47:29,691] Trial 2 finished with value: 0.21200000000000002 and parameters: {'ngram_range': '(1, 3)', 'max_features': None, 'binary': True, 'min_df': 1, 'max_df': 0.9, 'strip_accents': 

Best SVM Parameters: {'ngram_range': '(1, 4)', 'max_features': 10000, 'binary': False, 'min_df': 2, 'max_df': 0.9, 'strip_accents': 'unicode', 'lowercase': False, 'C': 0.01012588590893926, 'kernel': 'linear', 'class_weight': None}
Best SVM Accuracy: 0.8480000000000001
Best RF Parameters: {'ngram_range': '(1, 4)', 'max_features': None, 'binary': True, 'min_df': 1, 'max_df': 0.95, 'strip_accents': None, 'lowercase': False, 'n_estimators': 613, 'max_depth': 30, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features_rf': 'sqrt', 'bootstrap': False, 'class_weight': 'balanced'}
Best RF Accuracy: 0.8539999999999999


In [ ]:
svm_importances = optuna.importance.get_param_importances(svm_study)
print("SVM Hyperparameter Importance:")
for param, importance in svm_importances.items():
    print(f"{param}: {importance:.4f}")

rf_importances = optuna.importance.get_param_importances(rf_study)
print("\nRandom Forest Hyperparameter Importance:")
for param, importance in rf_importances.items():
    print(f"{param}: {importance:.4f}")

SVM Hyperparameter Importance:
kernel: 0.6677
min_df: 0.0938
ngram_range: 0.0590
lowercase: 0.0483
max_df: 0.0346
strip_accents: 0.0302
C: 0.0286
binary: 0.0198
max_features: 0.0126
class_weight: 0.0054

Random Forest Hyperparameter Importance:
class_weight: 0.2855
max_depth: 0.1764
max_features_rf: 0.1391
n_estimators: 0.1042
min_samples_leaf: 0.0980
min_df: 0.0583
lowercase: 0.0447
bootstrap: 0.0263
max_features: 0.0162
min_samples_split: 0.0151
max_df: 0.0136
strip_accents: 0.0124
ngram_range: 0.0069
binary: 0.0034


## Hyperparameter Optimization Results

We optimized **SVM** and **Random Forest** models using **Optuna**, tuning both **text preprocessing (CountVectorizer)** and **model parameters** to maximize classification accuracy.



## Hyperparameter Importance

###  SVM Key Parameters
| Parameter | Importance |
|------------|------------|
| Kernel | **0.6677** |
| min_df | 0.0938 |
| ngram_range | 0.0590 |
| C | 0.0286 |

**Insights:**  
- **Kernel type** has the biggest impact on performance.
- **Text preprocessing (min_df, ngram_range) also influences accuracy.**

---

###  Random Forest Key Parameters
| Parameter | Importance |
|------------|------------|
| Class Weight | **0.2855** |
| Max Depth | 0.1764 |
| Max Features | 0.1391 |
| n_estimators | 0.1042 |

**Insights:**  
- **Class balancing significantly improves RF performance.**
- **Tree depth and feature selection** are critical for accuracy.

---

##  Summary
- **SVM**: Kernel selection is the most important factor.
- **Random Forest**: Class weighting and tree depth are key.
- **Text preprocessing choices impact both models but to a lesser degree.**

 **These optimizations helped maximize model accuracy efficiently!**


We will now store the best hyperparameters with their fixed values so we dont have to run the whole optuna study each time the notebook restarts.

In [ ]:
best_svm_params = {'ngram_range': '(1, 4)', 'max_features': 10000, 'binary': False, 'min_df': 2, 'max_df': 0.9, 'strip_accents': 'unicode', 'lowercase': False, 'C': 0.01012588590893926, 'kernel': 'linear', 'class_weight': None}
best_rf_params = {'ngram_range': '(1, 4)', 'max_features': None, 'binary': True, 'min_df': 1, 'max_df': 0.95, 'strip_accents': None, 'lowercase': False, 'n_estimators': 613, 'max_depth': 30, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features_rf': 'sqrt', 'bootstrap': False, 'class_weight': 'balanced'}

We will try to use [cuml](https://github.com/rapidsai/cuml) to utilize gpu for SVM and RF

In [ ]:
!pip install \
    --extra-index-url=https://pypi.nvidia.com \
    "cudf-cu12==24.12.*" "dask-cudf-cu12==24.12.*" "cuml-cu12==24.12.*" \
    "cugraph-cu12==24.12.*" "nx-cugraph-cu12==24.12.*" "cuspatial-cu12==24.12.*" \
    "cuproj-cu12==24.12.*" "cuxfilter-cu12==24.12.*" "cucim-cu12==24.12.*" \
    "pylibraft-cu12==24.12.*" "raft-dask-cu12==24.12.*" "cuvs-cu12==24.12.*" \
    "nx-cugraph-cu12==24.12.*"


Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/26.9 MB 27.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 96.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.0/548.0 MB 50.0 MB/s eta 0:00:00:00:010:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.1/920.1 MB 23.8 MB/s eta 0:00:0000:01m0:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.0/152.0 kB 135.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 165.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.5/915.5 kB 65.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.6/83.6 kB 97.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 129.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 137.3 MB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.9/196.9 MB 85.5

Since the final kernel is linear, we could use LinearSVC optimized for linear kernels. However, LinearSVC does not support sparse vectors meaning that we would have to transform them to dense arrays. Turns out that this would just consume all the RAM of the system and beak. So we stay with regular SVM.

In [ ]:
import gc
import numpy as np
from cuml.svm import SVC
from cuml.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from tqdm import tqdm

We apply a CountVectorizer with the best parameters that we found during the hyperparameter tuning. This transforms the text data into numerical representations.

In [ ]:
vectorizer = CountVectorizer(
    ngram_range=eval(best_svm_params["ngram_range"]),  # Convert string back to tuple
    binary=best_svm_params["binary"],
    min_df=best_svm_params["min_df"],
    max_df=best_svm_params["max_df"],
    strip_accents=best_svm_params["strip_accents"],
    lowercase=best_svm_params["lowercase"]
)
X_vec = vectorizer.fit_transform(X).astype(np.float32)


We perform 5-Fold Cross-Validation using the GPU-accelerated SVM from cuML. The model is trained and evaluated across different splits to assess its generalization performance.

In [ ]:
svc_params = {
    'C': best_svm_params['C'],
    'kernel': best_svm_params['kernel'],
    'class_weight': best_svm_params['class_weight'],
    'verbose' : True
}

svm = SVC(**svc_params)


print("\nPerforming 5-Fold Cross-Validation for GPU-Accelerated SVM...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
svm_scores = []

for train_idx, test_idx in tqdm(kf.split(X_vec), desc="SVM Cross-Validation"):
    X_train, X_test = X_vec[train_idx], X_vec[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)
    svm_scores.append(accuracy_score(y_test, y_pred))

svm_scores = np.array(svm_scores)
print(f"\nSVM Cross-Validation Results: {svm_scores}")
print(f"SVM Mean Accuracy: {svm_scores.mean():.4f}, Std Dev: {svm_scores.std():.4f}")



Performing 5-Fold Cross-Validation for GPU-Accelerated SVM...


SVM Cross-Validation: 0it [00:00, ?it/s]

[D] [12:02:46.362698] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:02:57.061300] /__w/cuml/cuml/cpp/src/svm/smosolver.cuh:205 SMO solver finished after 50 outer iterations, total inner 24431 iterations, and diff 0.000943
[D] [12:02:57.170376] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:03:01.477627] /__w/cuml/cuml/cpp/src/svm/smosolver.cuh:205 SMO solver finished after 35 outer iterations, total inner 15506 iterations, and diff 0.000878
[D] [12:03:01.624549] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:03:41.120073] /__w/cuml/cuml/cpp/src/svm/smosolver.cuh:205 SMO solver finished after 132 outer iterations, total inner 72115 iterations, and diff 0.000990
[D] [12:03:41.251404] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:03:45.555633] /__w/cuml/cuml/cpp/src/svm/smosolver.cuh:205 SMO solver finished af

SVM Cross-Validation: 1it [01:48, 108.18s/it]

[D] [12:04:34.499303] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:04:45.496439] /__w/cuml/cuml/cpp/src/svm/smosolver.cuh:205 SMO solver finished after 50 outer iterations, total inner 23631 iterations, and diff 0.000944
[D] [12:04:45.606151] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:04:50.042935] /__w/cuml/cuml/cpp/src/svm/smosolver.cuh:205 SMO solver finished after 35 outer iterations, total inner 15247 iterations, and diff 0.000936
[D] [12:04:50.190015] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:05:31.199234] /__w/cuml/cuml/cpp/src/svm/smosolver.h:281 Solver is not converging monotonically. This might be caused by insufficient normalization of the feature columns. In that case MinMaxScaler((0,1)) could help. Alternatively, for nonlinear kernels, you can try to increase the gamma parameter. To limit execution time, you can also adjust the n

...

[D] [12:08:25.436209] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:08:36.822068] /__w/cuml/cuml/cpp/src/svm/smosolver.cuh:205 SMO solver finished after 51 outer iterations, total inner 24492 iterations, and diff 0.000909
[D] [12:08:36.932607] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:08:41.767221] /__w/cuml/cuml/cpp/src/svm/smosolver.cuh:205 SMO solver finished after 37 outer iterations, total inner 15950 iterations, and diff 0.000928
[D] [12:08:41.910419] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:09:24.819792] /__w/cuml/cuml/cpp/src/svm/smosolver.cuh:205 SMO solver finished after 129 outer iterations, total inner 73847 iterations, and diff 0.000999
[D] [12:09:24.953088] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:09:29.777102] /__w/cuml/cuml/cpp/src/svm/smosolver.cuh:205 SMO solver finished af

SVM Cross-Validation: 4it [07:34, 114.33s/it]

[D] [12:10:20.542365] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:10:32.106498] /__w/cuml/cuml/cpp/src/svm/smosolver.cuh:205 SMO solver finished after 48 outer iterations, total inner 23323 iterations, and diff 0.000886
[D] [12:10:32.217876] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:10:36.809611] /__w/cuml/cuml/cpp/src/svm/smosolver.cuh:205 SMO solver finished after 35 outer iterations, total inner 15658 iterations, and diff 0.000853
[D] [12:10:36.955912] /__w/cuml/cuml/cpp/src/svm/workingset.h:100 Creating working set with 1024 elements
[D] [12:11:17.160697] /__w/cuml/cuml/cpp/src/svm/smosolver.h:281 Solver is not converging monotonically. This might be caused by insufficient normalization of the feature columns. In that case MinMaxScaler((0,1)) could help. Alternatively, for nonlinear kernels, you can try to increase the gamma parameter. To limit execution time, you can also adjust the n

SVM Cross-Validation: 5it [09:30, 114.07s/it]


SVM Cross-Validation Results: [0.96600921 0.96757458 0.96583031 0.96596449 0.96905049]
SVM Mean Accuracy: 0.9669, Std Dev: 0.0013


## Results

- **SVM Cross-Validation Mean Accuracy**: `0.9669`
- **Standard Deviation**: `0.0013`
- The model performs consistently across different folds, with slight variations.

## Notes

- The GPU-accelerated SVM implementation gives as an  efficient training while maintaining high accuracy.
- The use of optimized hyperparameters increase our  model performance.

## Random Forest Cross-Validation

- **Model**: Random Forest Classifier with optimized hyperparameters.
- **Training Process**:
  We perform a **5-Fold Cross-Validation** to evaluate the model performance, using the best hyperparameters. Then we split the dataset into training and test set in each fold.After the split we train the model to predict test labels for each fold and append accuracy scores for evaluation.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=best_rf_params["n_estimators"],
    max_depth=best_rf_params["max_depth"],
    min_samples_split=best_rf_params["min_samples_split"],
    min_samples_leaf=best_rf_params["min_samples_leaf"],
    max_features=best_rf_params["max_features_rf"],
    bootstrap=best_rf_params["bootstrap"]
)

rf_scores = []


for train_idx, test_idx in tqdm(kf.split(X_vec), desc="Random Forest Cross-Validation"):
    X_train = X_vec[train_idx]
    X_test = X_vec[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    # Print shapes for debugging
    print(f"\nBatch shapes:")
    print(f"X_train: {X_train.shape}")
    print(f"y_train: {y_train.shape}")

    # Train and predict
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    rf_scores.append(accuracy_score(y_test, y_pred))

rf_scores = np.array(rf_scores)
print(f"\nRandom Forest Cross-Validation Results: {rf_scores}")
print(f"Random Forest Mean Accuracy: {rf_scores.mean():.4f}, Std Dev: {rf_scores.std():.4f}")

Random Forest Cross-Validation: 0it [00:00, ?it/s]


Batch shapes:
X_train: (89436, 7833527)
y_train: (89436,)


Random Forest Cross-Validation: 1it [30:20, 1820.18s/it]


Batch shapes:
X_train: (89436, 7833527)
y_train: (89436,)


Random Forest Cross-Validation: 2it [1:00:43, 1822.20s/it]

...


Batch shapes:
X_train: (89436, 7833527)
y_train: (89436,)


Random Forest Cross-Validation: 4it [2:00:42, 1806.99s/it]


Batch shapes:
X_train: (89436, 7833527)
y_train: (89436,)


Random Forest Cross-Validation: 5it [2:31:23, 1816.66s/it]


Random Forest Cross-Validation Results: [0.68692696 0.69309898 0.68057605 0.68598775 0.6969453 ]
Random Forest Mean Accuracy: 0.6887, Std Dev: 0.0057


## Results

- **Batch Shapes**:
  - `X_train: (89436, 7833527)`
  - `y_train: (89436,)`
- **Cross-Validation Progress**:
  - Iteration 1 completed in **30 minutes 27 seconds**.
  - Model performance remains consistent across iterations.

## Notes

- The Random Forest classifier successfully processes the large dataset.

Since SVM works better than the random forest, we will use SVM

## cuML SVM Training and Prediction

We perform **GPU-accelerated** SVM training and prediction using cuML’s **SVM** and **CountVectorizer**.

- **Key Components**:
  - **Vectorization**:  
    We utilize cuML’s `CountVectorizer` to transform the text data into numerical features. The configuration uses `ngram_range=(1,1)` and `binary=False`.
  - **SVM Model**:  
    We implement a **linear SVM** with `C=1.0` using cuML’s `SVC`, and we fit the model using the training data.
  - **Prediction Process**:  
    - The test data is transformed using the same `CountVectorizer`.
    - The trained SVM model is used to predict the labels for the test data.
    - The predictions are converted from cuDF to NumPy if needed for further analysis.




In [ ]:
vectorizer = CountVectorizer(ngram_range=(1, 1), binary=False)
X_vec = vectorizer.fit_transform(X)

X_test_vec = vectorizer.transform(X_test)

svm.fit(X_vec, y)

# Predict the labels for the test data
y_pred_test = svm.predict(X_test_vec)

# Convert predictions to NumPy array if needed
y_pred_test = y_pred_test.get()  # Convert from cuDF to NumPy array if using cuML

# Print or save the predictions
print(f"Predicted Labels for Test Data: {y_pred_test}")

We will now transform back the encoded labels to their original form and prepare the submission file.

We take the predicted labels, decodes them back to their original form using LabelEncoder, and create a DataFrame with the corresponding IDs. The predictions are then saved to a CSV file (predictions.csv) for further analysis or submission. Finally, a message confirms that the file has been successfully saved.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Assume y_pred_test is the predicted labels in encoded form
# and `label_encoder` was fitted on your original labels
decoded_predictions = label_encoder.inverse_transform(y_pred_test)

# Create a DataFrame with an Id column and Predicted column
predictions_df = pd.DataFrame({
    "Id": test_data['Id'],
    "Predicted": decoded_predictions
})

# Save the DataFrame to a CSV file
output_file = "predictions.csv"
predictions_df.to_csv(output_file, index=False)

print(f"Predictions saved to {output_file}")


Moving towards using a fine tuned transformer based model we will use the original data as it is, since those models have their own tokenizers to pre-process the input.

In [ ]:
import pandas as pd
train_data = pd.read_csv('/kaggle/input/bigdata2024classification/train.csv')

In [ ]:
test_data = pd.read_csv('/kaggle/input/bigdata2024classification/test_without_labels.csv')

Since this is actually a sequence classification model, we will use a BERT variant to fine tune. We will use DeBERTaV3 that, following ELECTRA, uses a different technique from MLM for trainig : Replaced Token Detection (RTD). Since the model can fit in our local GPU, which is 8GB, we will use this since it is faster than the one available from Kaggle.


We fine-tune a DeBERTa-v3 model for text classification. It starts by loading and preparing the dataset, where the Title and Content columns are combined into a single text field. The labels are encoded into numerical values to be used for training.

Next, a custom TextDataset class is created to handle the tokenization using  DeBERTa-v3 tokenizer. Both training and test datasets are processed accordingly.

The model is then fine-tuned with a learning rate of 5e-5, batch size of 16, and trained for one epoch using Hugging Face's Trainer API. After training, it makes predictions on the test data, converts the results back to their original labels, and saves them into a CSV file called test_predictions.csv.

We also use focal loss instead of cross entropy or whatever else, since our dataset is imnbalanced.

In [ ]:
!pip install sentence-transformers

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
from transformers import (
    DebertaV2Tokenizer, 
    DebertaV2ForSequenceClassification,
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
from torch.utils.data import Dataset
import torch
from torch.nn import CrossEntropyLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def clean_text(text):
    """Clean text by removing HTML tags and normalizing whitespace"""
    text = re.sub(r'<[^>]+>', '', str(text))  
    text = re.sub(r'\s+', ' ', text).strip()  
    return text

train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test_without_labels.csv')

model_name = "microsoft/deberta-v3-large"
tokenizer = DebertaV2Tokenizer.from_pretrained(model_name, use_fast=True)
sep_token = tokenizer.sep_token

train_data['Text'] = (train_data['Title'].apply(clean_text) + 
                     f" {sep_token} " + 
                     train_data['Content'].apply(clean_text))

test_data['Text'] = (test_data['Title'].apply(clean_text) + 
                    f" {sep_token} " + 
                    test_data['Content'].apply(clean_text))

label_encoder = LabelEncoder()
train_data['Label'] = label_encoder.fit_transform(train_data['Label'])

class_counts = train_data['Label'].value_counts().sort_index().values
class_weights = torch.tensor(1. / (class_counts / class_counts.sum()), 
                           dtype=torch.float32).to(device)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = self.tokenizer(
            self.texts[idx], 
            truncation=True, 
            max_length=self.max_len,  
            return_tensors="pt"
        )
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return {key: val.squeeze(0) for key, val in item.items()}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

class FocalLoss(torch.nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = torch.nn.functional.cross_entropy(
            inputs, targets, reduction='none', weight=self.weight.to(inputs.device)
        )
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() 

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get('logits')
        
        model_ref = model.module if hasattr(model, 'module') else model
        num_labels = model_ref.config.num_labels
        
        loss_fct = FocalLoss(weight=class_weights, gamma=2.0)
        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss
    
model = DebertaV2ForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-large",
    num_labels=len(label_encoder.classes_),
    attention_probs_dropout_prob=0.1,
    hidden_dropout_prob=0.2
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,  
    num_train_epochs=6,
    weight_decay=0.001,
    warmup_ratio=0.1,
    gradient_accumulation_steps=2,
    logging_dir='./logs',
    logging_steps=200,
    save_steps=1000,
    save_total_limit=2,
    eval_strategy="no", 
    report_to="none",
    dataloader_num_workers=4,
    lr_scheduler_type='cosine',
    gradient_checkpointing=True,
    ddp_find_unused_parameters=False,
    fp16=True, 
    dataloader_pin_memory=True,
    remove_unused_columns=False 
)
train_dataset = TextDataset(
    texts=train_data['Text'].tolist(),
    labels=train_data['Label'].tolist(),
    tokenizer=tokenizer,
    max_len=512
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)


trainer.train()

test_dataset = TextDataset(
    texts=test_data['Text'].tolist(),
    tokenizer=tokenizer,
    max_len=512  
)

trainer.model.train()  
num_tta = 3
logits_list = []

with torch.no_grad():
    for _ in range(num_tta):
        predictions = trainer.predict(test_dataset)
        logits_list.append(predictions.predictions)
        
final_logits = np.mean(logits_list, axis=0)
predicted_labels = np.argmax(final_logits, axis=1)

test_data['Predicted'] = label_encoder.inverse_transform(predicted_labels)
test_data[['Id', 'Predicted']].to_csv("test_predictions.csv", index=False)

print("Predictions saved to 'test_predictions.csv'")

# Exercise 2

The model was not evaluated using cross validation, since this is beyond the scope of this project. The reason we tried this, is to evaluate on the test set how better an advanced and larger model would work on this dataset. Turns out that the margin of improvement was negligible (smaller than 1%) at least in test set. #

In [ ]:
!pip install datasketch

In [ ]:
import pandas as pd
import time
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import pairwise_distances
from datasketch import MinHash, MinHashLSH
import numpy as np


Use the same pre-processed data as input.

In [ ]:
train_data = pd.read_pickle("/kaggle/input/pickle-files/processed_data.pkl")
test_data = pd.read_pickle("/kaggle/input/pickle-files/processed_test_data.pkl")

In [ ]:
train_data

,Id,Title,Content,Label,Text,Processed_Text
0,227464,"Netflix is coming to cable boxes, and Amazon i...",if you subscribe to one of three rinky-dink (...,Entertainment,"Netflix is coming to cable boxes, and Amazon i...",Netflix come cable box Amazon grocery overlord...
1,244074,"Pharrell, Iranian President React to Tehran 'H...","pharrell, iranian president react to tehran '...",Entertainment,"Pharrell, Iranian President React to Tehran 'H...",Pharrell iranian President React Tehran Happy ...
2,60707,Wildlife service seeks comments,the u.s. fish and wildlife service has reopen...,Technology,Wildlife service seeks comments the u.s. fish...,wildlife service seek comment u.s fish wildl...
3,27883,Facebook teams up with Storyful to launch 'FB ...,the very nature of social media means it is o...,Technology,Facebook teams up with Storyful to launch 'FB ...,facebook team Storyful launch FB Newswire na...
4,169596,Caesars plans US$880 mln New York casino,caesars plans us$880 mln new york casino jul ...,Business,Caesars plans US$880 mln New York casino caes...,Caesars plan us$ mln New York casino caesar ...
...,...,...,...,...,...,...
111790,31462,Microsoft requires Office 2013 licensing for s...,in contrast to the muckle of special licenses...,Technology,Microsoft requires Office 2013 licensing for s...,Microsoft require Office licensing non windows...
111791,100821,Smallpox vials missing since 1950s found in la...,government workers at a research center near ...,Health,Smallpox vials missing since 1950s found in la...,smallpox vial miss 1950 find lab storage room ...
111792,86181,Scientists May Have Just Discovered the Key to...,harvard scientists may have just unlocked the...,Health,Scientists May Have Just Discovered the Key to...,scientist discover Key reverse Aging harvard...
111793,256423,Justin Bieber to plead guilty to DUI,"justin bieber to plead guilty to duifri, 13 ju...",Entertainment,Justin Bieber to plead guilty to DUI justin bi...,Justin Bieber plead guilty DUI justin bieber p...


In [ ]:
test_data

,Id,Title,Content,Text,Processed_Text
0,262120,Tracy Morgan upgraded to fair condition after ...,actor and comedian tracy morgan has been upgr...,Tracy Morgan upgraded to fair condition after ...,Tracy Morgan upgrade fair condition crash ac...
1,175132,Smartphones Weigh on Samsung Electronics as Gu...,samsung electronics co ltd on tuesday issued u...,Smartphones Weigh on Samsung Electronics as Gu...,smartphone Weigh Samsung Electronics Guidance ...
2,218739,FBI denies fumbling testimony on 'X-Men' direc...,michael f. egan iii said in a press conferenc...,FBI denies fumbling testimony on 'X-Men' direc...,FBI deny fumble testimony X man director Bryan...
3,253483,Bachelorette 2014 Spoilers: Week 3 Recap ??? E...,i am having mixed emotions for what is about ...,Bachelorette 2014 Spoilers: Week 3 Recap ??? E...,Bachelorette spoiler week recap Eric Hill Dram...
4,224109,Barack Obama honours Frankie Knuckles in lette...,u.s. president barack obama has paid a specia...,Barack Obama honours Frankie Knuckles in lette...,barack Obama honour Frankie Knuckles letter lo...
...,...,...,...,...,...
47907,50348,"BMW, Tesla meet to discuss standardizing elect...","june 16, 2014 by edward taylor reutersan emplo...","BMW, Tesla meet to discuss standardizing elect...",BMW Tesla meet discuss standardize electric ca...
47908,255044,Harrison Ford has been filming the seventh Sta...,he may have helped save the galaxy from the ev...,Harrison Ford has been filming the seventh Sta...,Harrison Ford film Star Wars film Pinewood hel...
47909,66502,"It's Games, Games, Games As Microsoft Plans To...",less than three months after microsoft had a ...,"It's Games, Games, Games As Microsoft Plans To...",Games Games Games Microsoft plan close Xbox TV...
47910,10319,App Detail » Microsoft Excel for iPad,app description *** excel is ready for ipad p...,App Detail » Microsoft Excel for iPad app des...,App Detail Microsoft Excel iPad app descript...


# k-Nearest Neighbors (k-NN) with Jaccard Distance

We perform **k-NN classification** using **Jaccard distance** as a similarity metric for text data. It includes both a **vectorized** and a **non-vectorized** brute-force approach.

## Jaccard Distance Function
- Computes the **Jaccard similarity** between text feature sets.
- Uses **sparse matrices** for efficient storage and computation.
- Calculates:
  - **Intersection** common words between two samples.
  - **Union** total unique words in both samples.
- Ensures division by zero is avoided in distance computation.

## Brute-Force k-NN (Vectorized)
- **Optimized for speed** using **vectorized operations**.
- Uses **CountVectorizer** to convert text into binary word presence vectors.
- Processes **test data in batches** to **reduce memory usage**.
- Splits **train data into chunks** for **efficient distance computation**.
- Finds **k nearest neighbors** using **efficient sorting** techniques.

## Brute-Force k-NN (Non-Vectorized)
- A **simpler but slower approach** using **pairwise distances**.
- Converts train and test text data into **dense NumPy arrays**.
- Computes Jaccard similarity using **scikit-learn’s pairwise distance function**.
- Finds **k nearest neighbors** by directly sorting the distance matrix.

## Performance Considerations
- **Vectorized k-NN** is significantly **faster** and **more memory-efficient**.
- **Batch processing** prevents excessive memory usage on large datasets.
- **Chunking** allows us handling **large-scale** datasets smoothly.
- **Non-vectorized k-NN** is easier to understand but **computationally expensive**.

In [ ]:
def jaccard_distance(X, Y):

    # Compute intersection (element-wise multiplication of sparse matrices)
    intersection = X.dot(Y.T).toarray()

    # Compute union (vectorized)
    x_sum = X.sum(axis=1).A1
    y_sum = Y.sum(axis=1).A1
    # Broadcasting to compute union sizes
    union = x_sum[:, np.newaxis] + y_sum[np.newaxis, :] - intersection

    # Compute Jaccard distance (vectorized)
    # Handle division by zero
    union_nonzero = union != 0
    distances = np.ones_like(intersection, dtype=np.float32)
    distances[union_nonzero] = 1.0 - (intersection[union_nonzero] / union[union_nonzero])

    return distances

def brute_force_knn(train_data, test_data, k=7, batch_size=100):

    vectorizer = CountVectorizer(binary=True)
    train_matrix = vectorizer.fit_transform(train_data['Processed_Text'])

    start_time = time.time()
    n_test_samples = len(test_data)
    knn_indices = np.zeros((n_test_samples, k), dtype=int)

    for i in range(0, n_test_samples, batch_size):
        batch_end = min(i + batch_size, n_test_samples)
        batch_test = test_data['Processed_Text'].iloc[i:batch_end]

        test_batch_matrix = vectorizer.transform(batch_test)

        chunk_size = 5000
        batch_distances = []

        for j in range(0, train_matrix.shape[0], chunk_size):
            train_chunk = train_matrix[j:min(j + chunk_size, train_matrix.shape[0])]
            chunk_distances = jaccard_distance(test_batch_matrix, train_chunk)
            batch_distances.append(chunk_distances)

        # Combine chunks and find k nearest neighbors
        if len(batch_distances) > 1:
            batch_distances = np.hstack(batch_distances)
        else:
            batch_distances = batch_distances[0]

        batch_knn = np.argpartition(batch_distances, k, axis=1)[:, :k]

        # Sort the k neighbors by distance
        rows = np.arange(batch_distances.shape[0])[:, np.newaxis]
        batch_knn = batch_knn[rows, np.argsort(batch_distances[rows, batch_knn])]

        # Store the results
        knn_indices[i:batch_end] = batch_knn

        # Clean up
        del batch_distances
        del test_batch_matrix

    end_time = time.time()
    query_time = end_time - start_time

    return knn_indices, query_time

def brute_force_knn_non_vectorized(train_data, test_data, k=7):
    vectorizer = CountVectorizer(binary=True)
    train_matrix = vectorizer.fit_transform(train_data['Processed_Text']).toarray()
    test_matrix = vectorizer.transform(test_data['Processed_Text']).toarray()

    start_time = time.time()
    distances = pairwise_distances(test_matrix, train_matrix, metric='jaccard')
    knn_indices = distances.argsort(axis=1)[:, :k]
    end_time = time.time()

    query_time = end_time - start_time
    return knn_indices, query_time


# k-Nearest Neighbors (k-NN) with Locality-Sensitive Hashing (LSH)

We implement **k-NN classification** using **MinHash and LSH** to efficiently find similar text samples. This method is **faster than brute-force k-NN**, especially for large datasets.

## MinHash Function  
- Generates **MinHash signatures** for text data to approximate **Jaccard similarity**.  
- Splits text into **tokens** and updates the MinHash object.  
- Outputs a **MinHash signature** that represents the text.  

## LSH-Based k-NN  

### Building the LSH Index  
- Initializes an **LSH index** with a **Jaccard similarity threshold**.  
- Computes **MinHash signatures** for each training text.  
- Inserts these signatures into the **LSH index** for fast lookup.  

### Querying the LSH Index  
- Computes **MinHash signatures** for test samples.  
- Retrieves **candidate nearest neighbors** from the LSH index.  
- Computes **Jaccard similarity** between test and candidate samples.  
- Sorts candidates by similarity and selects the **k-nearest neighbors**.  

## Performance Considerations  
- **LSH k-NN is significantly faster** than brute-force k-NN for large datasets.  
- **Only relevant candidates are compared**, reducing computational cost.  
- **Threshold-based filtering** allows for quick elimination of dissimilar samples.  
- Best suited for **approximate nearest neighbor search** in large-scale text datasets.  


In [ ]:
def compute_minhash(text, num_perm):
    m = MinHash(num_perm=num_perm)
    tokens = text.split()
    for token in tokens:
        m.update(token.encode('utf8'))
    return m


def lsh_knn(train_data, test_data, k=7, num_perm=16, threshold=0.9):
    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)

    # Build LSH index
    start_build = time.time()
    train_minhashes = {}
    for i, text in enumerate(train_data['Processed_Text']):
        minhash = compute_minhash(text, num_perm)
        lsh.insert(str(i), minhash)
        train_minhashes[str(i)] = minhash
    end_build = time.time()

    # Query LSH index
    start_query = time.time()
    knn_indices = []
    for text in test_data['Processed_Text']:
        test_minhash = compute_minhash(text, num_perm)
        candidates = lsh.query(test_minhash)
        candidate_distances = [(c, 1 - train_minhashes[c].jaccard(test_minhash)) for c in candidates]
        candidate_distances.sort(key=lambda x: x[1])
        knn_indices.append([int(c[0]) for c in candidate_distances[:k]])
    end_query = time.time()

    build_time = end_build - start_build
    query_time = end_query - start_query

    return knn_indices, build_time, query_time

In [ ]:
k = 7
brute_knn, brute_query_time = brute_force_knn(train_data, test_data, k=k)

In [ ]:
# 1) Perform k-NN majority-vote classification
test_knn_predictions = []
for i in range(len(test_data)):
    # Grab the 7 training neighbor indices for test example i
    neighbors = brute_knn[i]  # shape (7,)

    # Retrieve the labels of these neighbors from the training set
    # .iloc[neighbors] picks those rows by index
    neighbor_labels = train_data.iloc[neighbors].Label

    # Find the most frequent label (majority vote)
    predicted_label = neighbor_labels.value_counts().idxmax()
    test_knn_predictions.append(predicted_label)

# 2) Build a result DataFrame with the columns "Id" and "Predicted"
results_df = pd.DataFrame({
    "Id": test_data["Id"],           # test IDs
    "Predicted": test_knn_predictions
})

# 3) Save to CSV (no index column in the CSV)
results_df.to_csv("knn_predictions.csv", index=False)

print("CSV saved: knn_predictions.csv")


CSV saved: knn_predictions.csv


It is worth noticing, that in test set in Kaggle competition KNN classifier using brute-force is the most accurate model, even outperforms the results of the fine-tuned DeBERTa model.

In [ ]:
#brute_knn_non_vectorized, brute_query_time_non_vectorized = brute_force_knn_non_vectorized(train_data, test_data, k=k)

brute_force_knn_non_vectorized cannot run, it consumes all available memory and kernel restarts

We will now split the training data to create a validation set. We will not use cross validation this time since the results are taking too long (and it is not explicitly required from the description). We will evaluate the time and accuracy on this validaiton set of brute force and LSH implementations.

In [ ]:
from sklearn.model_selection import train_test_split
train_split, val_split = train_test_split(train_data, test_size=0.2, random_state=42)


In [ ]:
k = 7
brute_knn, brute_query_time = brute_force_knn(train_split, val_split, k=k)


In [ ]:
k = 7
num_perms = [16, 32, 64]
thresh = 0.9

In [ ]:
# LSH
results = []
for num_perm in num_perms:
    lsh_knn_result, build_time, query_time = lsh_knn(
        train_split,
        val_split,
        k=k,
        num_perm=num_perm,
        threshold=thresh
    )

    # 1) Overlap with Brute Force
    # Make sure brute_knn_result and lsh_knn_result have same shape
    # (They should: both (# of validation docs, 7).)
    common_knn = 0
    for i in range(len(val_split)):
        # set(...) intersection set(...) => count
        common_knn += len(
            set(lsh_knn_result[i]).intersection(set(brute_knn[i]))
        )

    total_matches = len(val_split) * k
    fraction_overlap = common_knn / total_matches

    # 2) Accuracy via majority-vote classification
    # We'll compute a predicted label for each val row
    pred_labels = []
    for i in range(len(val_split)):
        # Top-7 neighbor indices from train_split
        neighbors_idx = lsh_knn_result[i]
        # Retrieve neighbor labels from train_split
        neighbor_labels = train_split.iloc[neighbors_idx].Label.values
        # Majority vote
        # .value_counts().idxmax() is one approach
        # Or we do np.bincount if numeric, etc.
        # Below is a quick approach:
        predicted_label = pd.Series(neighbor_labels).value_counts().idxmax()
        pred_labels.append(predicted_label)

    # Actual labels in the validation set
    true_labels = val_split.Label.values

    # Calculate accuracy
    correct_count = sum(
        1 for i in range(len(val_split)) if pred_labels[i] == true_labels[i]
    )
    accuracy = correct_count / len(val_split)

    # 3) Store metrics
    results.append({
        "Type": f"LSH-Jaccard-{num_perm}",
        "BuildTime": build_time,
        "QueryTime": query_time,
        "TotalTime": build_time + query_time,
        "FractionOverlap": fraction_overlap,
        "Accuracy": accuracy
    })

ValueError: The number of bands are too small (b < 2)

0.9 threshold cannot work : ValueError: The number of bands are too small (b < 2)


Also tried for threshold 0.8, 0.5 and 0.2, for the first two options the mean length of the list of the indices predicted by the MinHashLSH was smaller than 5, meaning that the LSH index did not retrieve more that 7 nearest neighbors on average.

In [ ]:
k = 7
num_perms = [16, 32, 64]
thresh = 0.2



We use  **LSH-based k-Nearest Neighbors (k-NN)**. The goal is to compare the efficiency and accuracy of **LSH Jaccard Similarity** against **Brute Force k-NN**

###  Key Steps:
1. **Iterate over different `num_perm` values**  
   - The loop runs for different values of `num_perm` to evaluate how the number of hash functions affects performance and accuracy.

2. **Perform LSH-based k-NN Search**  
   - The `lsh_knn` function is executed for each `num_perm` value.
   - This function returns:
     - `lsh_knn_result`: The k-NN results using LSH.
     - `build_time`: Time taken to construct the LSH index.
     - `query_time`: Time taken to perform k-NN queries.

3. **Compute Overlap with Brute Force k-NN**  
   - The fraction of LSH-based neighbors that also appear in the brute-force k-NN results is calculated.
   - This is done by **counting the common neighbors** between the brute force and LSH approaches for each validation document.

4. **Perform Classification using Majority Voting**  
   - For each validation document, the **labels of its k-NN neighbors** from `train_split` are retrieved.
   - The **majority class** among these neighbors is assigned as the predicted label.
   - If no neighbors are found (which is rare but possible), the label **defaults to "Entertainment"**.

5. **Evaluate Accuracy on the Validation Set**  
   - The predicted labels are compared against the actual labels in `val_split`.
   - Accuracy is computed as:  
     \[
     \text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Validation Samples}}
     \]

6. **Store Performance Metrics**  
   - The following performance metrics are recorded for each `num_perm` setting:
     - **Type**: Identifier for the LSH variant used.
     - **BuildTime**: Time taken to construct the LSH index.
     - **QueryTime**: Time taken for k-NN retrieval.
     - **TotalTime**: Sum of BuildTime and QueryTime.
     - **FractionOverlap**: The proportion of neighbors that LSH correctly retrieves compared to brute-force k-NN.
     - **Accuracy**: Classification accuracy on the validation set.


It helps analyze the **trade-off between speed and accuracy** in approximate nearest neighbor search using LSH. By adjusting `num_perm`, we can balance precision and computational efficiency.


In [ ]:
# LSH
results = []
for num_perm in num_perms:
    lsh_knn_result, build_time, query_time = lsh_knn(
        train_split,
        val_split,
        k=k,
        num_perm=num_perm,
        threshold=thresh
    )

    # 1) Overlap with Brute Force
    # Make sure brute_knn_result and lsh_knn_result have same shape
    # (They should: both (# of validation docs, 7).)
    common_knn = 0
    for i in range(len(val_split)):
        # set(...) intersection set(...) => count
        common_knn += len(
            set(lsh_knn_result[i]).intersection(set(brute_knn[i]))
        )

    total_matches = len(val_split) * k
    fraction_overlap = common_knn / total_matches

    # 2) Accuracy via majority-vote classification
    # We'll compute a predicted label for each val row
    pred_labels = []
    for i in range(len(val_split)):
        # Top-7 neighbor indices from train_split
        neighbors_idx = lsh_knn_result[i]
        # Retrieve neighbor labels from train_split
        neighbor_labels = train_split.iloc[neighbors_idx].Label.values
        # Majority vote
        if len(neighbor_labels) == 0:
            predicted_label = "Entertainment"
        else:
            predicted_label = pd.Series(neighbor_labels).value_counts().idxmax()
        pred_labels.append(predicted_label)

    # Actual labels in the validation set
    true_labels = val_split.Label.values

    # Calculate accuracy
    correct_count = sum(
        1 for i in range(len(val_split)) if pred_labels[i] == true_labels[i]
    )
    accuracy = correct_count / len(val_split)

    # 3) Store metrics
    results.append({
        "Type": f"LSH-Jaccard-{num_perm}",
        "BuildTime": build_time,
        "QueryTime": query_time,
        "TotalTime": build_time + query_time,
        "FractionOverlap": fraction_overlap,
        "Accuracy": accuracy
    })

In [ ]:
mean_length = sum(len(lst) for lst in lsh_knn_result) / len(lsh_knn_result)
print(f"The mean length of the lists is: {mean_length}")

The mean length of the lists is: 6.991681202200456


As we observe, the mean length is close to 7, meaning that our index returns >= candidate per query, so we are good

In [ ]:
pred_labels_brute = []
for i in range(len(val_split)):
    neighbors_idx = brute_knn[i]
    neighbor_labels = train_split.iloc[neighbors_idx].Label.values
    predicted_label = pd.Series(neighbor_labels).value_counts().idxmax()
    pred_labels_brute.append(predicted_label)

In [ ]:
true_labels = val_split.Label.values
correct_count = sum(
    1 for i in range(len(val_split)) if pred_labels_brute[i] == true_labels[i]
)
accuracy_brute = correct_count / len(val_split)

In [ ]:
results.append({
    "Type": "Brute-Force-Jaccard",
    "BuildTime": 0,
    "QueryTime": brute_query_time,
    "TotalTime": brute_query_time,
    "FractionOverlap": 1.0,  # by definition, brute force vs brute force
    "Accuracy": accuracy_brute
})

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df

,Type,BuildTime,QueryTime,TotalTime,FractionOverlap,Accuracy
0,LSH-Jaccard-16,104.882245,766.493124,871.375370,0.207484,0.834876
1,LSH-Jaccard-32,108.528672,77.745583,186.274255,0.284622,0.884297
2,LSH-Jaccard-64,115.121768,127.353200,242.474968,0.406177,0.932063
3,Brute-Force-Jaccard,0.000000,155.322470,155.322470,1.000000,0.965025


We now implement again the creation of MinHashLSH this time now using MinHashLsh forest, a function provided by datasketch that enavles us to perform queries in the MinHashLSH index without specifying a threshold, but using the number of K nearest neighbors to be returned.

In [ ]:
from datasketch import MinHashLSHForest, MinHash


def lsh_knn_forest(train_data, test_data, k=7, num_perm=16):
    lsh = MinHashLSHForest(num_perm=num_perm)

    # Build LSH index
    start_build = time.time()
    train_minhashes = {}
    for i, text in enumerate(train_data['Processed_Text']):
        minhash = compute_minhash(text, num_perm)
        lsh.add(str(i), minhash)
        train_minhashes[str(i)] = minhash
    lsh.index()
    end_build = time.time()

    # Query LSH index
    start_query = time.time()
    knn_indices = []
    for text in test_data['Processed_Text']:
        test_minhash = compute_minhash(text, num_perm)
        candidates = lsh.query(test_minhash,k)
        candidate_distances = [(c, 1 - train_minhashes[c].jaccard(test_minhash)) for c in candidates]
        candidate_distances.sort(key=lambda x: x[1])
        knn_indices.append([int(c[0]) for c in candidate_distances[:k]])
    end_query = time.time()

    build_time = end_build - start_build
    query_time = end_query - start_query

    return knn_indices, build_time, query_time





In the previous step, we defined the function `lsh_knn_forest`, which utilizes **MinHashLSHForest** from the `datasketch` library. This method improves upon the traditional MinHashLSH approach by allowing us to **query for the k-nearest neighbors directly**, without needing a predefined threshold.


1. **Building the LSH Forest Index**:
   - We compute the **MinHash signatures** for each document in `train_data`.
   - These signatures are stored in the **LSH Forest index**.
   - The index is then finalized using `lsh.index()` to optimize retrieval.

2. **Querying the LSH Forest**:
   - For each document in `test_data`, we compute its MinHash signature.
   - We then query the **LSH Forest** for the **k most similar documents** from `train_data`.
   - The retrieved neighbors are **sorted by Jaccard similarity** to ensure ranking quality.

3. **Computational Efficiency**:
   - The algorithm records the **build time** (time to construct the index).
   - It also records the **query time** (time to retrieve k-NN results).
   - Both of these metrics are useful for evaluating efficiency.

Now  we move to the **next step**, where we:
- **Compare the results** with brute-force k-NN.
- **Evaluate the overlap and accuracy** of the retrieved neighbors.
- **Store and analyze performance metrics**.

---


In [ ]:
for num_perm in num_perms:
    lsh_knn_result, build_time, query_time = lsh_knn_forest(
        train_split,
        val_split,
        k=k,
        num_perm=num_perm    )

    # 1) Overlap with Brute Force
    # Make sure brute_knn_result and lsh_knn_result have same shape
    # (They should: both (# of validation docs, 7).)
    common_knn = 0
    for i in range(len(val_split)):
        # set(...) intersection set(...) => count
        common_knn += len(
            set(lsh_knn_result[i]).intersection(set(brute_knn[i]))
        )

    total_matches = len(val_split) * k
    fraction_overlap = common_knn / total_matches

    # 2) Accuracy via majority-vote classification
    # We'll compute a predicted label for each val row
    pred_labels = []
    for i in range(len(val_split)):
        # Top-7 neighbor indices from train_split
        neighbors_idx = lsh_knn_result[i]
        # Retrieve neighbor labels from train_split
        neighbor_labels = train_split.iloc[neighbors_idx].Label.values
        # Majority vote
        if len(neighbor_labels) == 0:
            predicted_label = "Entertainment"
        else:
            predicted_label = pd.Series(neighbor_labels).value_counts().idxmax()
        pred_labels.append(predicted_label)

    # Actual labels in the validation set
    true_labels = val_split.Label.values

    # Calculate accuracy
    correct_count = sum(
        1 for i in range(len(val_split)) if pred_labels[i] == true_labels[i]
    )
    accuracy = correct_count / len(val_split)

    # 3) Store metrics
    results.append({
        "Type": f"LSH-Jaccard-Forest-{num_perm}",
        "BuildTime": build_time,
        "QueryTime": query_time,
        "TotalTime": build_time + query_time,
        "FractionOverlap": fraction_overlap,
        "Accuracy": accuracy
    })

In [ ]:
mean_length = sum(len(lst) for lst in lsh_knn_result) / len(lsh_knn_result)
print(f"The mean length of the lists is: {mean_length}")

The mean length of the lists is: 6.999642202245181


In [ ]:
results_df = pd.DataFrame(results)
results_df

,Type,BuildTime,QueryTime,TotalTime,FractionOverlap,Accuracy
0,LSH-Jaccard-16,104.882245,766.493124,871.375370,0.207484,0.834876
1,LSH-Jaccard-32,108.528672,77.745583,186.274255,0.284622,0.884297
2,LSH-Jaccard-64,115.121768,127.353200,242.474968,0.406177,0.932063
3,Brute-Force-Jaccard,0.000000,155.322470,155.322470,1.000000,0.965025
4,LSH-Jaccard-Forest-16,100.207206,25.581184,125.788390,0.047740,0.623597
5,LSH-Jaccard-Forest-32,107.136461,28.505409,135.641870,0.105474,0.725703
6,LSH-Jaccard-Forest-64,115.022427,33.759455,148.781882,0.100567,0.720515


Turns out that while the the query time is faster, the accuracy drops. This is normal because in the forest case we only rely on MinHash approximation of Jaccard similarity to get th K nearest neighbors and do not perform a finer filtering of the results returned by the MinHashLSH index by measurin the exact Jaccard similarity.

##  Optimized MinHashLSH k-NN with Vectorization

This implementation enhances **LSH-based k-Nearest Neighbors (k-NN)** by **vectorizing** key operations to improve efficiency. The goal is to  reducing redundant computations.


1. **Vectorized MinHash Computation**:
   - Instead of processing texts individually, we use **Scikit-learn’s `CountVectorizer`** to efficiently transform text data into tokenized features.
   - A single **encoded vocabulary** is created to avoid repeated encoding.
   - The MinHash signatures are computed in a **batch-wise manner**, reducing redundant loops.

2. **Efficient Jaccard Similarity Computation**:
   - Instead of computing **one-by-one** comparisons, we use a **vectorized** approach to compute similarities across multiple candidates **simultaneously**.

3. **Batch Processing for k-NN Search**:
   - The test dataset is processed in **batches** to avoid memory overload.
   - MinHash signatures for the test set are computed **in bulk**, reducing computational overhead.
   - The nearest neighbors are retrieved and **sorted efficiently** to select the top-k neighbors.

---



###  `vectorized_compute_minhashes`

Efficiently compute MinHash signatures for a batch of text documents.


- Uses `CountVectorizer` to tokenize text and generate **binary presence indicators**.
- Pre-encodes the **entire vocabulary** once (avoiding redundant encoding inside loops).
- Updates MinHash **only for present tokens** (instead of all vocabulary words).



---

### **`batch_jaccard_similarities`**

Efficiently compute Jaccard similarities **for multiple candidates at once**.


-We take a **list of candidate indices** and their respective MinHashes.
- Computes **Jaccard distances** in a **single loop** using NumPy operations.




---

###  `vectorized_lsh_knn`

Performs **LSH-based k-NN** search using a **fully vectorized approach**.

1.
   - Computes **all training MinHashes** in a **single pass**.
   - Inserts them into **MinHashLSH** for indexing.

2.
   - Processes test data in **batches** to optimize memory usage.
   - Uses `vectorized_compute_minhashes` to **batch-process** MinHashes for test documents.
   - Queries LSH for **nearest neighbors**, followed by **Jaccard similarity calculations**.

3.
   - Sorts retrieved candidates **efficiently**.
   - Selects **top-k neighbors**.

 **Stored Metrics**:
- `build_time`: Time taken to construct the LSH index.
- `query_time`: Time taken to retrieve neighbors.
- `knn_indices`: List of **top-k nearest neighbors** for each test document.
.



In [ ]:
import numpy as np
from datasketch import MinHash, MinHashLSH
import time
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer

def vectorized_compute_minhashes(texts, num_perm):
    """
    Vectorized computation of MinHash signatures for a batch of texts.

    Parameters:
    -----------
    texts : list of str
        List of text documents
    num_perm : int
        Number of permutations for MinHash

    Returns:
    --------
    list of MinHash : List of MinHash objects
    """
    # Vectorize text processing
    vectorizer = CountVectorizer(binary=True)
    text_matrix = vectorizer.fit_transform(texts)
    vocabulary = vectorizer.get_feature_names_out()

    # Pre-allocate list for MinHash objects
    minhashes = []

    # Create encoded tokens once
    encoded_vocab = [word.encode('utf8') for word in vocabulary]

    # Process each document using vectorized operations
    for i in range(text_matrix.shape[0]):
        m = MinHash(num_perm=num_perm)
        # Get indices of non-zero elements (tokens present in document)
        doc_tokens = text_matrix[i].indices
        # Update MinHash with present tokens
        for idx in doc_tokens:
            m.update(encoded_vocab[idx])
        minhashes.append(m)

    return minhashes

def batch_jaccard_similarities(candidates, train_minhashes, test_minhash):
    """
    Compute Jaccard similarities for multiple candidates in a vectorized way.

    Parameters:
    -----------
    candidates : list
        List of candidate indices
    train_minhashes : dict
        Dictionary of training set MinHash objects
    test_minhash : MinHash
        MinHash object for test document

    Returns:
    --------
    numpy.ndarray : Array of (candidate_idx, similarity) tuples
    """
    # Vectorized computation of Jaccard similarities
    similarities = np.array([
        (c, 1 - train_minhashes[c].jaccard(test_minhash))
        for c in candidates
    ])
    return similarities

def vectorized_lsh_knn(train_data, test_data, k=7, num_perm=16, threshold=0.9, batch_size=1000):
    """
    Vectorized implementation of LSH-based k-nearest neighbors search.

    Parameters:
    -----------
    train_data : pandas DataFrame
        Training data containing 'Processed_Text' column
    test_data : pandas DataFrame
        Test data containing 'Processed_Text' column
    k : int
        Number of nearest neighbors to find
    num_perm : int
        Number of permutations for MinHash
    threshold : float
        LSH threshold for candidate selection
    batch_size : int
        Size of batches for processing test data
    """
    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)

    # Build LSH index with vectorized MinHash computation
    start_build = time.time()
    train_texts = train_data['Processed_Text'].tolist()
    train_minhashes = {}

    # Compute all training MinHashes at once
    all_train_minhashes = vectorized_compute_minhashes(train_texts, num_perm)

    # Insert into LSH index
    for i, minhash in enumerate(all_train_minhashes):
        idx = str(i)
        lsh.insert(idx, minhash)
        train_minhashes[idx] = minhash

    end_build = time.time()

    # Query LSH index in batches
    start_query = time.time()
    knn_indices = []
    test_texts = test_data['Processed_Text'].tolist()
    n_test = len(test_texts)

    # Process test data in batches
    for i in range(0, n_test, batch_size):
        batch_end = min(i + batch_size, n_test)
        batch_texts = test_texts[i:batch_end]

        # Compute MinHashes for the batch
        batch_minhashes = vectorized_compute_minhashes(batch_texts, num_perm)

        # Process each test document in the batch
        for test_minhash in batch_minhashes:
            candidates = lsh.query(test_minhash)

            if not candidates:
                # If no candidates found, append k zeros as placeholder
                knn_indices.append([0] * k)
                continue

            # Compute similarities for all candidates at once
            candidate_distances = batch_jaccard_similarities(candidates, train_minhashes, test_minhash)

            # Sort by distance and get top k
            sorted_indices = candidate_distances[candidate_distances[:, 1].argsort()]
            top_k = sorted_indices[:k] if len(sorted_indices) >= k else np.pad(
                sorted_indices,
                ((0, k - len(sorted_indices)), (0, 0)),
                mode='constant'
            )

            knn_indices.append([int(c) for c, _ in top_k])

    end_query = time.time()

    build_time = end_build - start_build
    query_time = end_query - start_query

    return knn_indices, build_time, query_time

In [ ]:
for num_perm in num_perms:
    lsh_knn_result, build_time, query_time = vectorized_lsh_knn(
        train_split,
        val_split,
        k=k,
        num_perm=num_perm,
        threshold=thresh
    )

    # 1) Overlap with Brute Force
    # Make sure brute_knn_result and lsh_knn_result have same shape
    # (They should: both (# of validation docs, 7).)
    common_knn = 0
    for i in range(len(val_split)):
        # set(...) intersection set(...) => count
        common_knn += len(
            set(lsh_knn_result[i]).intersection(set(brute_knn[i]))
        )

    total_matches = len(val_split) * k
    fraction_overlap = common_knn / total_matches

    # 2) Accuracy via majority-vote classification
    # We'll compute a predicted label for each val row
    pred_labels = []
    for i in range(len(val_split)):
        # Top-7 neighbor indices from train_split
        neighbors_idx = lsh_knn_result[i]
        # Retrieve neighbor labels from train_split
        neighbor_labels = train_split.iloc[neighbors_idx].Label.values
        # Majority vote
        if len(neighbor_labels) == 0:
            predicted_label = "Entertainment"
        else:
            predicted_label = pd.Series(neighbor_labels).value_counts().idxmax()
        pred_labels.append(predicted_label)

    # Actual labels in the validation set
    true_labels = val_split.Label.values

    # Calculate accuracy
    correct_count = sum(
        1 for i in range(len(val_split)) if pred_labels[i] == true_labels[i]
    )
    accuracy = correct_count / len(val_split)

    # 3) Store metrics
    results.append({
        "Type": f"LSH-Jaccard-Vectorized-{num_perm}",
        "BuildTime": build_time,
        "QueryTime": query_time,
        "TotalTime": build_time + query_time,
        "FractionOverlap": fraction_overlap,
        "Accuracy": accuracy
    })

In [ ]:
results_df = pd.DataFrame(results)
results_df

,Type,BuildTime,QueryTime,TotalTime,FractionOverlap,Accuracy
0,LSH-Jaccard-16,104.882245,766.493124,871.375370,0.207484,0.834876
1,LSH-Jaccard-32,108.528672,77.745583,186.274255,0.284622,0.884297
2,LSH-Jaccard-64,115.121768,127.353200,242.474968,0.406177,0.932063
3,Brute-Force-Jaccard,0.000000,155.322470,155.322470,1.000000,0.965025
4,LSH-Jaccard-Forest-16,100.207206,25.581184,125.788390,0.047740,0.623597
5,LSH-Jaccard-Forest-32,107.136461,28.505409,135.641870,0.105474,0.725703
6,LSH-Jaccard-Forest-64,115.022427,33.759455,148.781882,0.100567,0.720515
7,LSH-Jaccard-Vectorized-16,83.471392,1119.028336,1202.499728,0.208832,0.834563
8,LSH-Jaccard-Vectorized-32,93.745594,105.133399,198.878993,0.287056,0.880496
9,LSH-Jaccard-Vectorized-64,102.501998,205.146688,307.648686,0.411391,0.929782


The results are the same, while the query time is higher, however the memory usage is really smaller, about 30% smaller than the original implementation.

# Exercise 3

Import necessary libraries

In [ ]:
import numpy as np
import pandas as pd
import time

Define the euclidean distance as the exercise demands

In [ ]:
def euclidean_distance(p1, p2):
    return np.sqrt((p1 - p2) ** 2)

Define the dynamic time wrapping function

In [ ]:
def dynamic_time_warping(seq_a, seq_b):
    matrix = np.zeros((len(seq_a) + 1, len(seq_b) + 1))
    matrix[0,:] = np.inf
    matrix[:,0] = np.inf
    matrix[0,0] = 0
    for i, v1 in enumerate(seq_a):
        for j, v2 in enumerate(seq_b):
            cost = euclidean_distance(v1, v2)
            matrix[i+1, j+1] = cost + min(matrix[i, j+1], matrix[i+1, j], matrix[i, j])
    return matrix[-1,-1]


In [ ]:
data = pd.read_csv('/kaggle/input/timeseriesexercise3/dtw_test.csv')
results = []
start = time.time()
for i,row in data.iterrows():
    seq_a = eval(row['series_a'])  # Convert string to list
    seq_b = eval(row['series_b'])
    distance = dynamic_time_warping(seq_a, seq_b)
    results.append({'id': row['id'], 'DTW distance': distance})
end = time.time()

In [ ]:
total_time = end - start
print(f"Total time to compute DTW on the dataset: {total_time} seconds")

Total time to compute DTW on the dataset: 1173.691163778305 seconds


In [ ]:
results_df = pd.DataFrame(results)
results_df

,id,DTW distance
0,0,18.875200
1,1,16.439700
2,2,12.072300
3,3,17.015000
4,4,8.736900
...,...,...
997,997,14.431400
998,998,5.447000
999,999,0.727379
1000,1000,2.237100
